# OrbitLab: CUDA N-body gravity
Select a GPU runtime. Clone the public repository below. Each cell invokes repository programs; lessons and equations are in README/docs. Temporary Colab files disappear on runtime replacement.

In [ ]:
from pathlib import Path
import subprocess,os,json
for command in (["nvidia-smi","--query-gpu=name,compute_cap","--format=csv,noheader"],["nvcc","--version"]):
    print(subprocess.check_output(command,text=True))


## Acquire sources from GitHub
Clone the public repository into a fresh directory. To reproduce a particular revision, check out its commit before building.

In [ ]:
root=Path("/content/orbitlab-clean")
assert not root.exists(),"Use a fresh runtime or a new empty directory"
subprocess.run(["git","clone","https://github.com/setsam37/orbitlab-cuda-nbody.git",str(root)],check=True)
assert (root/"CMakeLists.txt").exists()
print("Source revision:",subprocess.check_output(["git","rev-parse","HEAD"],cwd=root,text=True).strip())


In [ ]:
def run(command):
    result=subprocess.run(list(map(str,command)),cwd=root,env=os.environ,capture_output=True,text=True)
    print(result.stdout+result.stderr)
    result.check_returncode()
    return result
run(["python","-m","pip","install","-r","requirements.txt"])
cap=subprocess.check_output(["nvidia-smi","--query-gpu=compute_cap","--format=csv,noheader"],text=True).splitlines()[0].strip().replace(".","")
run(["cmake","-S",".","-B","build-gpu","-DCMAKE_BUILD_TYPE=Release","-DORBITLAB_ENABLE_CUDA=ON",f"-DCMAKE_CUDA_ARCHITECTURES={cap}"])
run(["cmake","--build","build-gpu","--parallel","2"])
exe=root/"build-gpu/orbitlab"
os.environ["ORBITLAB_EXE"]=str(exe)


In [ ]:
run(["ctest","--test-dir","build-gpu","--output-on-failure"])
run(["python","-m","unittest","discover","-s","scripts","-p","test_*.py"])


In [ ]:
run(["python","scripts/run_science.py","--exe",exe,"--output","results/science"])
run(["python","scripts/plot_science.py","--input","results/science"])
run(["python","scripts/animate.py","--trajectory","results/science/cuda-tiled-float/trajectory.csv","--output","results/orbit.gif"])


## Performance experiment
This sweeps 120 configurations, including partial tiles and precision choices. On the tested T4 runtime this takes minutes. Timing variability is expected; inspect batch ranges, not just one ratio.

In [ ]:
run(["python","scripts/run_benchmarks.py","--exe",exe,"--output","results/benchmarks"])
run(["python","scripts/plot_benchmarks.py","--input","results/benchmarks"])
run(["python","scripts/profile_kernels.py","--exe",exe,"--output","results/profiling"])


In [ ]:
from IPython.display import display,Image
print(json.loads((root/"results/science/summary.json").read_text()))
display(Image(filename=str(root/"results/orbit.gif")))
display(Image(filename=str(root/"results/science/convergence.png")))


## Keep the results
Download before leaving Colab. The notebook saves no fabricated execution outputs; local verified logs/results accompany the project.

In [ ]:
import shutil
archive=shutil.make_archive("/content/orbitlab-run-results","zip",root,"results")
files.download(archive)
